# Typed Wire Layer

> Typed data transfer at the worker boundary — the zero-copy `FileBackedDTO`
> protocol and (stage 2 of the post-pass-2 execution sequence) the typed
> result envelope that carries task results across the worker HTTP/JSON
> boundary, retiring the per-consumer `field_of` dict-or-object tolerance
> helpers (pass-2 Thread 3; evidence E5/D10/C7).

In [ ]:
#| default_exp core.wire

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Protocol, runtime_checkable

## FileBackedDTO Protocol

The `FileBackedDTO` protocol defines objects that can serialize themselves to disk for zero-copy transfer between Host and Worker processes. When the Proxy detects an argument implementing this protocol, it calls `to_temp_file()` and sends the file path instead of the data.

In [ ]:
#| export
@runtime_checkable
class FileBackedDTO(Protocol):
    """Protocol for Data Transfer Objects that serialize to disk for zero-copy transfer."""
    
    def to_temp_file(self) -> str: # Absolute path to the temporary file
        """Save the data to a temporary file and return the absolute path."""
        ...

In [ ]:
# Test FileBackedDTO protocol detection
import tempfile

class MockAudioData:
    """Example class implementing FileBackedDTO."""
    
    def __init__(self, data: bytes):
        self._data = data
    
    def to_temp_file(self) -> str:
        """Save to temp file and return path."""
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
            f.write(self._data)
            return f.name

# Check if it implements the protocol
audio = MockAudioData(b"fake audio data")
print(f"MockAudioData implements FileBackedDTO: {isinstance(audio, FileBackedDTO)}")
print(f"Temp file path: {audio.to_temp_file()}")

# A regular string does not implement the protocol
print(f"str implements FileBackedDTO: {isinstance('hello', FileBackedDTO)}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()